In [ ]:
import pandas as pd
import numpy as np
import mlflow.sklearn
from sklearn.preprocessing import LabelEncoder

mlflow.set_tracking_uri("https://dagshub.com/Sula1909/ML-Assignment1.mlflow")

model = mlflow.sklearn.load_model("models:/HousePrices_XGBoost/1")

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

train = train.drop(train[(train['GrLivArea']>4000) & (train['SalePrice']<300000)].index)

ntrain = train.shape[0]
test_id = test['Id']
y_train_orig = train.SalePrice.values

train.drop("Id", axis=1, inplace=True)
test.drop("Id", axis=1, inplace=True)

def preprocess_data(df):
    df = df.copy()
    if 'Utilities' in df.columns:
        df = df.drop(['Utilities'], axis=1)

    none_cols = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 
                 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType', 'PoolQC', 
                 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'MSSubClass']
    for col in none_cols:
        df[col] = df[col].fillna("None")

    zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 
                 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
    for col in zero_cols:
        df[col] = df[col].fillna(0)

    df["Functional"] = df["Functional"].fillna("Typ")
    mode_cols = ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType']
    for col in mode_cols:
        df[col] = df[col].fillna(df[col].mode()[0])

    df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(lambda x: x.fillna(x.median()))

    for col in ['MSSubClass', 'OverallCond', 'YrSold', 'MoSold']:
        df[col] = df[col].astype(str)

    cols = ('FireplaceQu', 'BsmtQual', 'BsmtCond', 'GarageQual', 'GarageCond', 
            'ExterQual', 'ExterCond','HeatingQC', 'PoolQC', 'KitchenQual', 'BsmtFinType1', 
            'BsmtFinType2', 'Functional', 'Fence', 'BsmtExposure', 'GarageFinish', 'LandSlope',
            'LotShape', 'PavedDrive', 'Street', 'Alley', 'CentralAir', 'MSSubClass', 'OverallCond', 
            'YrSold', 'MoSold')
    for c in cols:
        lbl = LabelEncoder()
        lbl.fit(list(df[c].values))
        df[c] = lbl.transform(list(df[c].values))

    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df = pd.get_dummies(df)
    return df

all_data = pd.concat((train.drop(['SalePrice'], axis=1), test)).reset_index(drop=True)
all_data_processed = preprocess_data(all_data)

train_final = all_data_processed[:ntrain]
test_final = all_data_processed[ntrain:]

temp_train = train_final.copy()
temp_train['SalePrice'] = np.log1p(y_train_orig)
correlations = temp_train.corr()['SalePrice'].abs().sort_values(ascending=False)
selected_features = correlations[correlations > 0.1].index.tolist()
selected_features.remove('SalePrice')

test_final_fs = test_final[selected_features]

log_preds = model.predict(test_final_fs)
final_preds = np.expm1(log_preds)

submission = pd.DataFrame({'Id': test_id, 'SalePrice': final_preds})
submission.to_csv("submission.csv", index=False)

print("XGBoost Submission Generated Successfully!")
print(submission.head())

XGBoost Submission Generated Successfully!
     Id      SalePrice
0  1461  125954.484375
1  1462  156696.484375
2  1463  188069.343750
3  1464  191892.296875
4  1465  195188.375000
